# Reusable TensorBoard Projector Pipeline (v1 - Modular Version)

## Intro

This notebook introduces a refactored, reusable version of the embedding visualization pipeline.

It improves on the earlier `build_projector_v0.ipynb` by:
 - supporting multiple datasets with a single function
 - eliminating duplicated logic
 - making output fully configurable via `out_prefix`
 - ensuring consistent artifact generation (embeddings, metadata, sprite)

**What this notebook achieves**
It generalizes the pipeline:
```
images → embeddings → metadata + sprite → TensorBoard Projector files
```
into a reusable function:
```
build_projector_files(image_root, out_prefix)
```

**Prerequisite**

Before running this notebook, ensure you executed:
```
notebooks/prepare_data.ipynb
```

This guarantees:
 - consistent folder structure
 - clean filenames
 - limited dataset size (for digits)


## Step 1 - Build reusable embedding pipeline

We define a reusable function that processes any image dataset folder.

In [5]:
from pathlib import Path
from PIL import Image
import torch
import torchvision.models as models
import csv
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = models.ResNet18_Weights.DEFAULT
base_model = models.resnet18(weights=weights)

# Remove classification head → 512-d embeddings
model = torch.nn.Sequential(*list(base_model.children())[:-1]).to(device).eval()

# Use official preprocessing (critical for correctness)
transform = weights.transforms()

In [6]:
def build_projector_files(image_root: str, out_prefix: str):

    # ---------------------------
    # Step 1: Collect images
    # ---------------------------
    image_paths = sorted(
        [
            p for p in Path(image_root).rglob("*")
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ]
    )

    if not image_paths:
        raise ValueError(f"No images found in {image_root}")

    # ---------------------------
    # Step 2: Extract embeddings
    # ---------------------------
    def get_vec(p):
        with Image.open(p) as img:
            img = img.convert("RGB")
            batch = transform(img).unsqueeze(0).to(device)

            with torch.no_grad():
                vec = model(batch)

        return vec.squeeze().cpu().numpy()

    vecs = [get_vec(p) for p in image_paths]

    with open(f"../vis/{out_prefix}_feature_vecs.tsv", "w") as f:
        csv.writer(f, delimiter="\t").writerows(vecs)

    # ---------------------------
    # Step 3: Metadata
    # ---------------------------
    with open(f"../vis/{out_prefix}_metadata.tsv", "w") as f:
        w = csv.writer(f, delimiter="\t")
        w.writerow(["label", "file"])

        for p in image_paths:
            w.writerow([p.parent.name, p.name])

    # ---------------------------
    # Step 4: Sprite image
    # ---------------------------
    images = []
    for p in image_paths:
        with Image.open(p) as img:
            images.append(img.resize((100, 100)))

    w, h = images[0].size
    grid = int(np.ceil(np.sqrt(len(images))))
    sprite = Image.new("RGB", (w * grid, h * grid))

    for i, img in enumerate(images):
        r, c = divmod(i, grid)
        sprite.paste(img, (c * w, r * h))

    sprite.save(f"../vis/{out_prefix}_sprite.jpg")

## Step 2 - Run for all datasets

Make sure data preparation has been executed first.

In [7]:
build_projector_files("../images", "animals")
build_projector_files("../digits", "digits")

## Step 3 - TensorBoard Projector config

This file tells TensorBoard how to load embeddings.

Create:
`vis/projector_config.pbtxt`

```
embeddings {
  tensor_name: "animals_resnet18"
  tensor_path: "animals_feature_vecs.tsv"
  metadata_path: "animals_metadata.tsv"
  sprite {
    image_path: "animals_sprite.jpg"
    single_image_dim: [100, 100]
  }
}

embeddings {
  tensor_name: "digits_resnet18"
  tensor_path: "digits_feature_vecs.tsv"
  metadata_path: "digits_metadata.tsv"
  sprite {
    image_path: "digits_sprite.jpg"
    single_image_dim: [100, 100]
  }
}
```

## Step 4 - Run TensorBoard

```
tensorboard --logdir ./vis
```

Then open:
```
http://localhost:6006
```


## Summary

**Key design improvements over v0**
 - Reusability - One function works for all datasets
 - Dataset abstraction - No hardcoded paths or structures
 - Scalability - Easy to add new datasets: `build_projector_files("../cats_vs_dogs", "cats_dogs")`
 - Deterministic output

This notebook represents the bridge between experimentation and production code.

Final production pipeline:
```
prepare_data.py
    ↓
build_projector.py
    ↓
TensorBoard Projector
```

